# Knowledge Flow Application on Data Sets — Weka/R Style Demonstration

This notebook reproduces the three experiments described in the assignment using Python:

1. **Strong association rules** using **Apriori** and **FP-Growth**.
2. **ARFF batch loading + 10-fold cross-validation** using a **J48-style decision tree**.
3. **Multiple ROC curves in one plot** for a J48-style tree and Random Forest.

> **Note:** Weka Knowledge Flow is a graphical workflow environment. The notebook below provides an executable Python equivalent of the same workflow. The tree experiment uses scikit-learn's entropy-based decision tree as a practical J48-style implementation; exact Weka J48 results can differ slightly because Weka's J48 is C4.8 with its own pruning/details.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations
from collections import defaultdict
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score
)

np.random.seed(42)
print("Libraries imported successfully.")


## 1. Create an ARFF-style transaction dataset

The Weka workflow normally starts with `ArffLoader`. To make this notebook self-contained, the code below creates a small transaction dataset and writes it as an ARFF file. The same file is then loaded into pandas.


In [ ]:
# Transaction data for association-rule mining
transactions = [
    ["Milk", "Bread", "Butter"],
    ["Milk", "Bread"],
    ["Milk", "Diapers", "Beer", "Bread"],
    ["Bread", "Butter"],
    ["Milk", "Diapers", "Beer"],
    ["Milk", "Bread", "Diapers", "Beer"],
    ["Bread", "Butter", "Eggs"],
    ["Milk", "Bread", "Butter", "Eggs"],
    ["Milk", "Diapers", "Beer", "Bread"],
    ["Bread", "Butter", "Milk"],
    ["Milk", "Bread", "Eggs"],
    ["Diapers", "Beer"],
    ["Milk", "Bread", "Butter"],
    ["Bread", "Eggs"],
    ["Milk", "Diapers", "Bread", "Beer"]
]

items = ["Beer", "Bread", "Butter", "Diapers", "Eggs", "Milk"]

# Convert transactions to 0/1 item columns
basket = pd.DataFrame(
    [{item: int(item in t) for item in items} for t in transactions]
)

basket.index.name = "Transaction"
print(basket)
print("\nTransactions:", len(basket), " | Items:", len(items))


In [ ]:
# Write the transaction data to ARFF format, similar to Weka's ArffLoader input.
arff_path = "association_transactions.arff"

with open(arff_path, "w", encoding="utf-8") as f:
    f.write("@relation association_transactions\n\n")
    for item in items:
        f.write(f"@attribute {item} {{0,1}}\n")
    f.write("\n@data\n")
    for row in basket[items].itertuples(index=False):
        f.write(",".join(map(str, row)) + "\n")

print("ARFF file created:", arff_path)

# Simple ARFF loader for this self-contained notebook
def load_arff_binary(path):
    data_started = False
    rows = []
    cols = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("%"):
                continue
            low = line.lower()
            if low.startswith("@attribute"):
                parts = line.split()
                cols.append(parts[1])
            elif low.startswith("@data"):
                data_started = True
            elif data_started:
                rows.append(line.split(","))
    return pd.DataFrame(rows, columns=cols).astype(int)

loaded_basket = load_arff_binary(arff_path)
print("\nLoaded using the ARFF loader equivalent:")
print(loaded_basket.head())


## 2. Discretization step

The original Knowledge Flow layout optionally places **Discretize** between `ArffLoader` and the associators. Association mining here uses binary transaction attributes, so no additional discretization is required. For continuous attributes, Weka's `Discretize` can convert numeric ranges into bins before Apriori/FP-Growth.


## 3. Apriori — find frequent itemsets and strong association rules

Parameters corresponding to common Weka settings:

- Minimum support: **0.20**
- Minimum confidence: **0.60**
- Number of rules requested: **10**

Support = transactions containing the complete itemset / total transactions.

Confidence(A → B) = support(A ∪ B) / support(A).


In [ ]:
def apriori(transactions, min_support=0.20, max_len=None):
    n = len(transactions)
    trans_sets = [set(t) for t in transactions]
    items_all = sorted(set().union(*trans_sets))
    frequent = {}

    # 1-itemsets
    current = {}
    for item in items_all:
        sup = sum(item in t for t in trans_sets) / n
        if sup >= min_support:
            current[frozenset([item])] = sup
    frequent.update(current)

    k = 2
    while current and (max_len is None or k <= max_len):
        prev_sets = list(current.keys())
        candidates = set()
        for a, b in combinations(prev_sets, 2):
            union = a | b
            if len(union) == k:
                # Apriori pruning: every (k-1)-subset must be frequent
                if all(frozenset(s) in current for s in combinations(union, k-1)):
                    candidates.add(union)

        next_current = {}
        for cand in candidates:
            sup = sum(cand.issubset(t) for t in trans_sets) / n
            if sup >= min_support:
                next_current[cand] = sup
        frequent.update(next_current)
        current = next_current
        k += 1
    return frequent

transaction_sets = [set(t) for t in transactions]
freq_ap = apriori(transaction_sets, min_support=0.20)

freq_table = pd.DataFrame([
    {"itemset": ", ".join(sorted(s)), "support": round(sup, 3), "count": round(sup*len(transactions))}
    for s, sup in freq_ap.items()
]).sort_values(["support", "itemset"], ascending=[False, True])

print("Frequent itemsets found:", len(freq_table))
display(freq_table)


In [ ]:
def association_rules_from_itemsets(freq, min_confidence=0.60, max_rules=10):
    rules = []
    for itemset, sup_xy in freq.items():
        if len(itemset) < 2:
            continue
        items_list = list(itemset)
        # all non-empty proper antecedents
        for r in range(1, len(items_list)):
            for antecedent_tuple in combinations(items_list, r):
                antecedent = frozenset(antecedent_tuple)
                consequent = itemset - antecedent
                if antecedent not in freq:
                    continue
                conf = sup_xy / freq[antecedent]
                if conf >= min_confidence:
                    lift = conf / freq[consequent]
                    rules.append({
                        "antecedent": ", ".join(sorted(antecedent)),
                        "consequent": ", ".join(sorted(consequent)),
                        "support": sup_xy,
                        "confidence": conf,
                        "lift": lift
                    })
    rules.sort(key=lambda x: (x["confidence"], x["support"], x["lift"]), reverse=True)
    return pd.DataFrame(rules[:max_rules])

ap_rules = association_rules_from_itemsets(freq_ap, 0.60, 10)
ap_rules.round(3)


### 4. FP-Growth — parallel association workflow

FP-Growth uses a compact FP-tree rather than generating candidate itemsets in the same way as Apriori. The Knowledge Flow idea is the same: send the loaded dataset to the FP-Growth associator and inspect the frequent itemsets/rules.


In [ ]:
# Self-contained FP-Growth implementation for this demonstration.

def build_fptree(transactions, min_count):
    header = {}
    root = {"item": None, "count": 0, "parent": None, "children": {}, "link": None}

    # Count items
    counts = defaultdict(int)
    for t in transactions:
        for item in t:
            counts[item] += 1
    counts = {i:c for i,c in counts.items() if c >= min_count}
    if not counts:
        return root, header

    for item in counts:
        header[item] = [counts[item], None]

    for t in transactions:
        ordered = sorted([i for i in t if i in counts],
                         key=lambda i: (-counts[i], str(i)))
        node = root
        for item in ordered:
            if item in node["children"]:
                child = node["children"][item]
                child["count"] += 1
            else:
                child = {"item": item, "count": 1, "parent": node,
                         "children": {}, "link": None}
                node["children"][item] = child
                if header[item][1] is None:
                    header[item][1] = child
                else:
                    cur = header[item][1]
                    while cur["link"] is not None:
                        cur = cur["link"]
                    cur["link"] = child
            node = child
    return root, header

def conditional_pattern_base(item, header):
    patterns = []
    node = header[item][1]
    while node is not None:
        path = []
        parent = node["parent"]
        while parent is not None and parent["item"] is not None:
            path.append(parent["item"])
            parent = parent["parent"]
        if path:
            patterns.append((path, node["count"]))
        node = node["link"]
    return patterns

def fp_growth_recursive(transactions, min_count, suffix=frozenset()):
    root, header = build_fptree(transactions, min_count)
    results = {}
    # Mine less frequent items first
    for item in sorted(header, key=lambda x: header[x][0]):
        new_itemset = suffix | frozenset([item])
        results[new_itemset] = header[item][0]
        patterns = conditional_pattern_base(item, header)
        conditional_transactions = []
        for path, count in patterns:
            for _ in range(count):
                conditional_transactions.append(path)
        if conditional_transactions:
            sub = fp_growth_recursive(conditional_transactions, min_count, new_itemset)
            results.update(sub)
    return results

min_support = 0.20
min_count = int(np.ceil(min_support * len(transactions)))
fp_counts = fp_growth_recursive(transaction_sets, min_count)
freq_fp = {k: v/len(transactions) for k, v in fp_counts.items()}

fp_table = pd.DataFrame([
    {"itemset": ", ".join(sorted(s)), "support": round(sup, 3), "count": round(sup*len(transactions))}
    for s, sup in freq_fp.items()
]).sort_values(["support", "itemset"], ascending=[False, True])

print("FP-Growth frequent itemsets found:", len(fp_table))
display(fp_table)


In [ ]:
fp_rules = association_rules_from_itemsets(freq_fp, 0.60, 10)
print("Top FP-Growth association rules:")
display(fp_rules.round(3))


### 5. Knowledge Flow layout — association rules

Conceptually, the Weka canvas is:

`ArffLoader → Discretize → Apriori → TextViewer`

and in parallel:

`Discretize → FP-Growth → TextViewer`

The notebook has reproduced the same data flow programmatically.


# 6. ARFF batch mode + Cross Validation + J48

The second workflow is:

`ArffLoader → ClassAssigner → CrossValidationFoldMaker → J48 → ClassifierPerformanceEvaluator → TextViewer`

This notebook uses a generated classification dataset and writes it as an ARFF file, then performs stratified 10-fold cross-validation.


In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=240,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42
)

feature_names = [f"feature_{i+1}" for i in range(X.shape[1])]
classification_df = pd.DataFrame(X, columns=feature_names)
classification_df["class"] = np.where(y == 1, "yes", "no")

classification_arff = "classification_data.arff"
with open(classification_arff, "w", encoding="utf-8") as f:
    f.write("@relation classification_data\n\n")
    for col in feature_names:
        f.write(f"@attribute {col} numeric\n")
    f.write("@attribute class {no,yes}\n\n@data\n")
    for _, row in classification_df.iterrows():
        vals = [f"{row[c]:.6f}" for c in feature_names] + [row["class"]]
        f.write(",".join(vals) + "\n")

print("ARFF batch file created:", classification_arff)
print(classification_df.head())


In [ ]:
# ClassAssigner equivalent: 'class' is the target attribute.
X_cls = classification_df[feature_names]
y_cls = classification_df["class"]

# J48-style classifier: entropy criterion + pruning-like max depth constraint.
# In Weka, J48 is C4.8; this is a practical Python equivalent.
j48 = DecisionTreeClassifier(
    criterion="entropy",
    random_state=42,
    max_depth=None,
    min_samples_leaf=2
)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
j48_pred = cross_val_predict(j48, X_cls, y_cls, cv=cv, method="predict")

acc = accuracy_score(y_cls, j48_pred)
cm = confusion_matrix(y_cls, j48_pred, labels=["no", "yes"])

print("10-Fold Cross Validation — J48-style Decision Tree")
print("Accuracy:", round(acc, 4))
print("\nConfusion Matrix [no, yes]:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_cls, j48_pred, digits=4))


### 7. Interpretation of the J48 cross-validation output

The evaluator reports:

- **Accuracy** — percentage of correctly classified instances.
- **Confusion matrix** — actual versus predicted class counts.
- **Precision, recall and F1-score** — per-class performance.
- The same workflow can be configured to use a different number of folds; **10 folds** is the usual default in Weka.


# 8. Plot multiple ROC curves: J48 and Random Forest

The third Knowledge Flow layout sends the same cross-validation data to two classifiers:

- `J48`
- `RandomForest`

Each classifier connects to its own `ClassifierPerformanceEvaluator`, and both evaluators send `thresholdData` to one `ModelPerformanceChart`.

This produces **two ROC curves in the same plot window**.


In [ ]:
# Cross-validated probability predictions for both models
j48_prob = cross_val_predict(
    j48, X_cls, y_cls, cv=cv, method="predict_proba"
)[:, 1]

rf = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    max_features="sqrt"
)

rf_pred = cross_val_predict(rf, X_cls, y_cls, cv=cv, method="predict")
rf_prob = cross_val_predict(
    rf, X_cls, y_cls, cv=cv, method="predict_proba"
)[:, 1]

j48_auc = roc_auc_score((y_cls == "yes").astype(int), j48_prob)
rf_auc = roc_auc_score((y_cls == "yes").astype(int), rf_prob)

print("J48-style Accuracy:", round(accuracy_score(y_cls, j48_pred), 4))
print("J48-style AUC:", round(j48_auc, 4))
print("Random Forest Accuracy:", round(accuracy_score(y_cls, rf_pred), 4))
print("Random Forest AUC:", round(rf_auc, 4))


In [ ]:
y_binary = (y_cls == "yes").astype(int)

fpr_j48, tpr_j48, _ = roc_curve(y_binary, j48_prob)
fpr_rf, tpr_rf, _ = roc_curve(y_binary, rf_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr_j48, tpr_j48, label=f"J48-style (AUC = {j48_auc:.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {rf_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — J48 and Random Forest")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 9. Complete Knowledge Flow summary

### Association rules
`ArffLoader → Discretize → {Apriori, FP-Growth} → TextViewer`

### J48 cross-validation
`ArffLoader → ClassAssigner → CrossValidationFoldMaker → J48 → ClassifierPerformanceEvaluator → TextViewer`

### Multiple ROC curves
`ArffLoader → ClassAssigner → CrossValidationFoldMaker`

then in parallel:

`→ J48 → ClassifierPerformanceEvaluator ┐`

`→ RandomForest → ClassifierPerformanceEvaluator ├→ ModelPerformanceChart`

The notebook is self-contained and does not require a separate Weka installation. For an exact Weka submission, the same layouts can be built in **Tools → Knowledge Flow** using the component names in the assignment.


## Expected output

After **Run All**, the notebook will display:

1. The generated ARFF transaction dataset.
2. Frequent itemsets and strong rules from Apriori.
3. Frequent itemsets and rules from FP-Growth.
4. A 10-fold J48-style cross-validation accuracy, confusion matrix and classification report.
5. J48-style and Random Forest AUC values.
6. **One plot window containing both ROC curves**.

Because the notebook creates its own deterministic dataset with `random_state=42`, the numerical classification results are reproducible.
